<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/02_ML_Engineer/beginner/03_docker_intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Docker for ML Engineers

Docker eliminates "works on my machine" by packaging your code, dependencies, and environment into a portable container. Every ML production system runs in containers.

**Topics:** Dockerfile, multi-stage builds, docker-compose, GPU support, optimizing ML images

## 1. Basic Dockerfile for ML Inference

In [ ]:
basic_dockerfile = """
# Dockerfile — basic ML inference server
FROM python:3.11-slim

# Security: don't run as root
RUN useradd -m -u 1000 appuser

# Install system dependencies
RUN apt-get update && apt-get install -y \\
    libgomp1 \\
    curl \\
    && rm -rf /var/lib/apt/lists/*  # clean up to reduce image size

WORKDIR /app

# Copy and install dependencies FIRST (cache optimization)
# Docker caches each layer — if requirements.txt unchanged, this layer is cached
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY src/ ./src/
COPY models/ ./models/

# Switch to non-root user
USER appuser

# Expose port
EXPOSE 8000

# Health check
HEALTHCHECK --interval=30s --timeout=5s --start-period=15s --retries=3 \\
    CMD curl -f http://localhost:8000/health || exit 1

# Start server
CMD ["uvicorn", "src.main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "4"]
"""

print('=== Basic ML Inference Dockerfile ===')
print(basic_dockerfile)

commands = """
Build and run:
  docker build -t my-ml-service:v1 .
  docker run -p 8000:8000 my-ml-service:v1
  
  # Run in background
  docker run -d --name ml-service -p 8000:8000 my-ml-service:v1
  
  # View logs
  docker logs ml-service -f
  
  # Shell into running container
  docker exec -it ml-service /bin/bash
  
  # Check resource usage
  docker stats ml-service
"""
print(commands)

## 2. Multi-Stage Build — Production Optimized

In [ ]:
multistage_dockerfile = """
# Dockerfile.prod — multi-stage build
# Stage 1: Builder — installs dependencies, compiles code
FROM python:3.11-slim AS builder

RUN apt-get update && apt-get install -y build-essential gcc

WORKDIR /build
COPY requirements.txt .
RUN pip install --no-cache-dir --prefix=/install -r requirements.txt

# Stage 2: Runtime — only what's needed to RUN the app
FROM python:3.11-slim AS runtime

# Copy installed packages from builder (no build tools needed at runtime!)
COPY --from=builder /install /usr/local

RUN apt-get update && apt-get install -y libgomp1 curl \\
    && rm -rf /var/lib/apt/lists/* \\
    && useradd -m -u 1000 appuser

WORKDIR /app
COPY src/ ./src/
COPY models/ ./models/

USER appuser
EXPOSE 8000
HEALTHCHECK --interval=30s --timeout=5s CMD curl -f http://localhost:8000/health || exit 1
CMD ["uvicorn", "src.main:app", "--host", "0.0.0.0", "--port", "8000"]

# Result: Builder image ~2GB, Runtime image ~400MB
#         Never ship build tools and compilers to production!
"""
print('=== Multi-Stage Dockerfile ===')
print(multistage_dockerfile)

## 3. Docker Compose for ML Stack

In [ ]:
docker_compose = """
# docker-compose.yml — full ML stack locally
version: '3.9'

services:
  # ML inference API
  ml-api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - MODEL_PATH=/models/xgboost_v2.pkl
      - DATABASE_URL=postgresql://postgres:postgres@db:5432/mldb
      - REDIS_URL=redis://redis:6379
    volumes:
      - ./models:/models:ro  # read-only model mount
      - ./logs:/app/logs
    depends_on:
      db:
        condition: service_healthy
      redis:
        condition: service_healthy
    restart: unless-stopped
    deploy:
      resources:
        limits:
          memory: 4G
          cpus: '2.0'
  
  # PostgreSQL for feature store / predictions
  db:
    image: postgres:15-alpine
    environment:
      POSTGRES_DB: mldb
      POSTGRES_USER: postgres
      POSTGRES_PASSWORD: postgres
    volumes:
      - postgres_data:/var/lib/postgresql/data
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U postgres"]
      interval: 10s
      timeout: 5s
      retries: 5
  
  # Redis for caching predictions
  redis:
    image: redis:7-alpine
    command: redis-server --maxmemory 512mb --maxmemory-policy allkeys-lru
    healthcheck:
      test: ["CMD", "redis-cli", "ping"]
      interval: 10s
  
  # MLflow tracking server
  mlflow:
    image: ghcr.io/mlflow/mlflow:latest
    ports:
      - "5000:5000"
    command: mlflow server --host 0.0.0.0 --backend-store-uri postgresql://postgres:postgres@db:5432/mlflow
    depends_on:
      - db

volumes:
  postgres_data:
"""
print('=== docker-compose.yml (Full ML Stack) ===')
print(docker_compose)

print("""
Usage:
  docker compose up -d          # start all services
  docker compose logs -f ml-api # follow API logs
  docker compose ps             # check status
  docker compose down           # stop all
  docker compose down -v        # stop + remove volumes
""")

## 4. GPU-Enabled Docker for Training

In [ ]:
gpu_dockerfile = """
# Dockerfile.gpu — GPU-enabled PyTorch training
# Use NVIDIA CUDA base image
FROM nvcr.io/nvidia/pytorch:24.01-py3

WORKDIR /workspace

# Install additional Python packages
COPY requirements-training.txt .
RUN pip install --no-cache-dir -r requirements-training.txt

COPY src/ ./src/
COPY configs/ ./configs/

# Default command: training script
CMD ["python", "-m", "src.train", "--config", "configs/train_config.yaml"]

# Run with GPU:
# docker run --gpus all -v $(pwd)/data:/data -v $(pwd)/checkpoints:/checkpoints \\
#   my-training:latest
"""

best_practices = """
=== Docker ML Best Practices ===

Image optimization:
  • Use slim/alpine base images (python:3.11-slim, not python:3.11)
  • Install requirements before copying code (layer caching)
  • Combine RUN commands with && to reduce layers
  • Use .dockerignore to exclude: __pycache__, .git, data/, notebooks/
  • Multi-stage builds: separate build and runtime stages
  • Final ML inference image target: < 1GB

Security:
  • Never run as root in production
  • Never hardcode secrets — use environment variables or secrets manager
  • Scan images: docker scout, trivy, snyk
  • Use specific image tags (python:3.11.7-slim), not latest

.dockerignore:
  __pycache__/
  *.pyc
  .git/
  data/
  notebooks/
  mlruns/
  *.pkl  # models loaded at runtime from volume/S3
  .env
  tests/

CUDA image selection (NVIDIA):
  nvcr.io/nvidia/pytorch:24.01-py3     → PyTorch + CUDA 12.3
  nvcr.io/nvidia/tensorflow:24.01-tf2-py3 → TF + CUDA 12.3
  nvcr.io/nvidia/cuda:12.3.2-base-ubuntu22.04 → bare CUDA
"""
print(gpu_dockerfile)
print(best_practices)